# TimeH2O on GIFT-Eval

[TimeH2O](https://huggingface.co/collections/h2oai/timeh2o) is a family of time-series foundation
models by H2O.ai: [TimeH2O-v1](https://huggingface.co/h2oai/timeh2o-v1) (108.8M parameters) and
[TimeH2O-v1-tiny](https://huggingface.co/h2oai/timeh2o-v1-tiny) (5.8M), served by the
[`h2ofm`](https://pypi.org/project/h2ofm/) package.

This notebook produced the `results/TimeH2O-v1` and `results/TimeH2O-v1-tiny` submissions, with
`gift_eval.data.Dataset` for the data and GluonTS `evaluate_model` for the metrics, as in the other notebooks.

Protocol: zero-shot; full history as context (the model keeps the last 16384 steps); multivariate
targets forecast independently and pooled in the metrics; `past_feat_dynamic_real` unused (no past
covariates in these checkpoints); `clip_nonnegative`, `snap_integer`, `flip_equivariant` (free, the
checkpoints are flip-equivariant) and `patch_ensemble` (TimeH2O-v1 only); float32; median point forecast.

## Setup

1. Install GIFT-Eval as instructed in the [README](../README.md).
2. Download the dataset: `hf download Salesforce/GiftEval --repo-type=dataset --local-dir /path/to/gifteval`
3. `pip install h2ofm` (Python >= 3.12). The checkpoints are gated: accept the license on their model pages and `hf auth login`.
4. Set `GIFT_EVAL` to the dataset path (see cell below).

The 97 configurations run sequentially by default. Several copies (one per GPU) sharing the same
`OUTPUT_DIR` split the work through the `claims/` folder.


In [ ]:
import json
import logging
import math
import os
import time
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from gluonts.ev.metrics import (
    MAE, MAPE, MASE, MSE, MSIS, ND, NRMSE, RMSE, SMAPE,
    MeanWeightedSumQuantileLoss,
)
from gluonts.model import evaluate_model
from gluonts.model.forecast import QuantileForecast
from gluonts.time_feature import get_seasonality

from gift_eval.data import Dataset
from h2ofm import TimeH2O

os.environ.setdefault("GIFT_EVAL", "Change/To/GiftEval/Local/Path")

logging.getLogger("gluonts.model.forecast").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)  # pandas 'H'/'T' freq aliases used by gift-eval

In [ ]:
# Environment variables let one copy of this notebook per GPU share the work; the defaults run everything here.
CHECKPOINT_NAME = os.environ.get("TIMEH2O_MODEL", "timeh2o-v1")  # timeh2o-v1 | timeh2o-v1-tiny
MODEL_NAME = {"timeh2o-v1": "TimeH2O-v1", "timeh2o-v1-tiny": "TimeH2O-v1-tiny"}[CHECKPOINT_NAME]
CHECKPOINT = f"h2oai/{CHECKPOINT_NAME}"  # downloaded from the Hugging Face Hub
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path(os.environ.get("TIMEH2O_OUTPUT_DIR", "outputs")) / MODEL_NAME
LANE = os.environ.get("TIMEH2O_LANE", "0")  # label for the log lines when several copies run

# Rows per model pass (targets, not series): these fit a 24 GB GPU at the 16384-step context (h2ofm's default is 256).
BATCH_SIZE = 64 if CHECKPOINT_NAME == "timeh2o-v1" else 128
# Series per forecast_many call: sorted by length so padding stays small.
MAX_SERIES = 1024
MAX_TOKENS = 2**24  # series x (context + horizon) per call
MAX_CONTEXT = 16384  # the checkpoints' context window; longer histories are trimmed by h2ofm anyway

FORECAST_OPTIONS = dict(
    clip_nonnegative=True,
    snap_integer=True,
    flip_equivariant=True,
    patch_ensemble=True,  # single-size checkpoints ignore it
)

In [ ]:
SHORT_DATASETS = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
MED_LONG_DATASETS = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"

PRETTY_DATASET_NAMES = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
METRICS = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=QUANTILE_LEVELS),
]
METRIC_COLUMNS = [
    "MSE[mean]", "MSE[0.5]", "MAE[0.5]", "MASE[0.5]", "MAPE[0.5]",
    "sMAPE[0.5]", "MSIS", "RMSE[mean]", "NRMSE[mean]", "ND[0.5]",
    "mean_weighted_sum_quantile_loss",
]


def all_configs(properties: dict) -> list[tuple[str, str, str]]:
    """(dataset_name, term, config key) for the 97 configs."""
    short_datasets = SHORT_DATASETS.split()
    med_long_datasets = MED_LONG_DATASETS.split()
    configs = []
    for dataset_name in dict.fromkeys(short_datasets + med_long_datasets):
        for term in ["short", "medium", "long"]:
            if term in ("medium", "long") and dataset_name not in med_long_datasets:
                continue
            if "/" in dataset_name:
                ds_key, ds_freq = dataset_name.split("/")
                ds_key = PRETTY_DATASET_NAMES.get(ds_key.lower(), ds_key.lower())
            else:
                ds_key = PRETTY_DATASET_NAMES.get(dataset_name.lower(), dataset_name.lower())
                ds_freq = properties[ds_key]["frequency"]
            configs.append((dataset_name, term, f"{ds_key}/{ds_freq}/{term}"))
    assert len(configs) == 97, len(configs)
    return configs

In [ ]:
class TimeH2OPredictor:
    """GluonTS-style predictor: predict(test_data.input) -> list[QuantileForecast].

    Series are sorted by length and forecast in groups within the MAX_SERIES / MAX_TOKENS budget
    (h2ofm batches the rows of a group BATCH_SIZE at a time). A group that runs out of GPU memory is
    split in two; a single series that still does not fit is retried without the patch ensemble, then
    with half the context, and the fallback is recorded.
    """

    def __init__(self, model: TimeH2O, prediction_length: int):
        self.model = model
        self.prediction_length = prediction_length
        self.forecast_keys = [str(q) for q in QUANTILE_LEVELS] + ["mean"]
        self.fallbacks: list[dict] = []

    def predict(self, test_data_input) -> list[QuantileForecast]:
        entries = list(test_data_input)
        contexts = []
        for e in entries:
            t = np.asarray(e["target"], dtype=np.float32)
            contexts.append((t[None] if t.ndim == 1 else t)[:, -MAX_CONTEXT:])  # (V, T)

        outputs = [None] * len(entries)
        order = sorted(range(len(entries)), key=lambda i: contexts[i].shape[-1])
        start = 0
        while start < len(order):
            # Grow the group from the shortest remaining series while the padded size fits the budget.
            end = start + 1
            while end < len(order):
                length = contexts[order[end]].shape[-1] + self.prediction_length
                if end + 1 - start > MAX_SERIES or (end + 1 - start) * length > MAX_TOKENS:
                    break
                end += 1
            self._predict_group(order[start:end], contexts, outputs)
            start = end

        forecasts = []
        for e, q in zip(entries, outputs):
            # q: (V, H, 9) -> (10, H, V), or (10, H) for a univariate target; "mean" is the median.
            arr = np.concatenate([q, q[..., QUANTILE_LEVELS.index(0.5), None]], axis=-1)
            arr = arr.transpose(2, 1, 0)
            if np.asarray(e["target"]).ndim == 1:
                arr = arr[..., 0]
            forecasts.append(
                QuantileForecast(
                    forecast_arrays=arr,
                    forecast_keys=self.forecast_keys,
                    start_date=e["start"] + np.asarray(e["target"]).shape[-1],
                    item_id=e.get("item_id"),
                )
            )
        return forecasts

    def _predict_group(self, idx, contexts, outputs, **overrides) -> None:
        try:
            q = self._run([contexts[i] for i in idx], **overrides)
        except torch.cuda.OutOfMemoryError:
            q = None  # handled below, outside the except block, so the failed pass's tensors are freed
        if q is not None:
            for j, i in enumerate(idx):
                outputs[i] = q[j]
            return
        torch.cuda.empty_cache()
        if len(idx) > 1:
            half = len(idx) // 2
            self._predict_group(idx[:half], contexts, outputs)
            self._predict_group(idx[half:], contexts, outputs)
            return
        if not overrides:
            overrides = dict(patch_ensemble=False)
        elif "context_length" not in overrides:
            overrides = dict(patch_ensemble=False, context_length=MAX_CONTEXT // 2)
        else:
            raise torch.cuda.OutOfMemoryError(f"item {idx[0]} does not fit even at half context")
        self.fallbacks.append({"item": int(idx[0]), **overrides})
        self._predict_group(idx, contexts, outputs, **overrides)

    def _run(self, group, **overrides) -> list[np.ndarray]:
        forecasts = self.model.forecast_many(
            group,
            self.prediction_length,
            quantile_levels=QUANTILE_LEVELS,
            **{**FORECAST_OPTIONS, **overrides},
        )
        return [f.quantiles for f in forecasts]  # each (V, H, 9)

In [ ]:
def evaluate_config(model: TimeH2O, dataset_name: str, term: str) -> tuple[dict, dict]:
    ds = Dataset(name=dataset_name, term=term, to_univariate=False)
    predictor = TimeH2OPredictor(model, ds.prediction_length)
    res = evaluate_model(
        predictor,
        test_data=ds.test_data,
        metrics=METRICS,
        batch_size=512,
        axis=None,
        mask_invalid_label=True,
        allow_nan_forecast=False,
        seasonality=get_seasonality(ds.freq),
    )
    info = {
        "prediction_length": ds.prediction_length,
        "windows": ds.windows,
        "target_dim": ds.target_dim,
        "fallbacks": predictor.fallbacks,
    }
    return {k: float(res[k].iloc[0]) for k in res.columns}, info

In [ ]:
# Runs the 97 configs, longest first. Each config's metrics go to OUTPUT_DIR/per_config/ as soon as it is
# done; finished configs are skipped, so a restart (or another copy of this notebook on another GPU) picks
# up where the work stopped. A copy claims a config by creating OUTPUT_DIR/claims/<config>.claim; a claim
# without a result after STALE_CLAIM_SECONDS is taken over. A config that fails MAX_ATTEMPTS times is
# recorded as <config>.failed and skipped.
STALE_CLAIM_SECONDS = 6 * 3600
MAX_ATTEMPTS = 3

properties = json.load(open("dataset_properties.json"))
model = TimeH2O.from_pretrained(CHECKPOINT, device=DEVICE, batch_size=BATCH_SIZE)
print(f"{CHECKPOINT} on {DEVICE}: {model.config}")

per_config_dir = OUTPUT_DIR / "per_config"
claims_dir = OUTPUT_DIR / "claims"
per_config_dir.mkdir(parents=True, exist_ok=True)
claims_dir.mkdir(parents=True, exist_ok=True)


def result_path(key: str) -> Path:
    return per_config_dir / (key.replace("/", "__") + ".json")


def work_estimate(dataset_name: str, term: str) -> float:
    ds = Dataset(name=dataset_name, term=term, to_univariate=False)
    return ds.windows * ds.target_dim * ds.sum_series_length


def claim(key: str) -> bool:
    path = claims_dir / (key.replace("/", "__") + ".claim")
    try:
        fd = os.open(path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError:
        if time.time() - path.stat().st_mtime < STALE_CLAIM_SECONDS:
            return False
        path.touch()  # stale: take it over
        return True
    os.write(fd, f"{LANE} {os.getpid()}\n".encode())
    os.close(fd)
    return True


def release(key: str) -> None:
    (claims_dir / (key.replace("/", "__") + ".claim")).unlink(missing_ok=True)


# Claims left by an earlier run of this copy (same LANE label) belong to a dead process: drop them.
for stale in claims_dir.glob("*.claim"):
    if stale.read_text().split()[0] == LANE:
        stale.unlink()

configs = all_configs(properties)
if os.environ.get("TIMEH2O_ONLY"):  # for testing: comma-separated config keys, e.g. "m4_hourly/H/short"
    configs = [c for c in configs if c[2] in os.environ["TIMEH2O_ONLY"].split(",")]
configs.sort(key=lambda c: work_estimate(c[0], c[1]), reverse=True)

for dataset_name, term, key in configs:
    path = result_path(key)
    failed = path.with_suffix(".failed")
    if path.exists() or failed.exists() or not claim(key):
        continue
    for attempt in range(1, MAX_ATTEMPTS + 1):
        t0 = time.time()
        try:
            metrics, info = evaluate_config(model, dataset_name, term)
        except Exception:
            print(f"[{LANE}] {key}: attempt {attempt} failed\n{traceback.format_exc()}")
            torch.cuda.empty_cache()
            if attempt == MAX_ATTEMPTS:
                failed.write_text(traceback.format_exc())
            continue
        info["seconds"] = time.time() - t0
        path.write_text(json.dumps({"dataset": key, "model": MODEL_NAME, "metrics": metrics, **info}, indent=1))
        print(
            f"[{LANE}] {key}: MASE={metrics['MASE[0.5]']:.4f} CRPS={metrics['mean_weighted_sum_quantile_loss']:.4f}"
            f" ({info['seconds']:.0f}s, fallbacks={len(info['fallbacks'])})"
        )
        break
    release(key)

done = len(list(per_config_dir.glob("*.json")))
print(f"{done}/97 configs done for {MODEL_NAME}")

In [ ]:
# Writes results/<MODEL_NAME>/all_results.csv in the leaderboard format once all 97 configs are done.
rows = []
for p in sorted(per_config_dir.glob("*.json")):
    r = json.loads(p.read_text())
    ds_key = r["dataset"].split("/")[0]
    rows.append(
        {
            "dataset": r["dataset"],
            "model": MODEL_NAME,
            **{f"eval_metrics/{k}": r["metrics"][k] for k in METRIC_COLUMNS},
            "domain": properties[ds_key]["domain"],
            "num_variates": properties[ds_key]["num_variates"],
        }
    )
complete = len(rows) == 97
if complete:
    results = pd.DataFrame(rows).sort_values("dataset")
    results_dir = Path("../results") / MODEL_NAME
    results_dir.mkdir(parents=True, exist_ok=True)
    results.to_csv(results_dir / "all_results.csv", index=False)
    print(f"Saved {len(results)} configs to {results_dir / 'all_results.csv'}")
else:
    print(f"{len(rows)} of 97 configs done; all_results.csv not written")

In [ ]:
# Leaderboard aggregates: MASE and CRPS of each config divided by Seasonal Naive, then the
# geometric mean over the 97 configs.
def gmean(x: pd.Series) -> float:
    return float(math.exp(x.map(math.log).mean()))


if complete:
    seasonal_naive = pd.read_csv("../results/Seasonal_Naive/all_results.csv").set_index("dataset")
    res = results.set_index("dataset").loc[seasonal_naive.index]
    for name, col in [("MASE", "eval_metrics/MASE[0.5]"), ("CRPS", "eval_metrics/mean_weighted_sum_quantile_loss")]:
        print(f"{name}: {gmean(res[col] / seasonal_naive[col]):.4f}")